In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------

# Create widgets for configurable parameters
dbutils.widgets.text("num_patients", "327", "Number of Patients")
dbutils.widgets.text("events_per_patient_max", "8", "Max Events Per Patient")
dbutils.widgets.text("hl7_version", "2.5", "HL7 Version")
dbutils.widgets.text("sending_app", "DATABRICKS_SIM", "Sending Application")
dbutils.widgets.text("sending_facility", "DBX_FAC", "Sending Facility")
dbutils.widgets.text("receiving_app", "DOWNSTREAM_SYS", "Receiving Application")
dbutils.widgets.text("receiving_facility", "DST_FAC", "Receiving Facility")
dbutils.widgets.text("catalog_use", "main", "Catalog Name")
dbutils.widgets.text("schema_use", "healthcare", "Schema Name")
dbutils.widgets.text("volume_name", "hl7_synthetic", "Volume Name")
dbutils.widgets.text("relative_path", "adt_fake", "Relative Path")

# Get widget values
num_patients = int(dbutils.widgets.get("num_patients"))
events_per_patient_max = int(dbutils.widgets.get("events_per_patient_max"))
hl7_version = dbutils.widgets.get("hl7_version")
sending_app = dbutils.widgets.get("sending_app")
sending_facility = dbutils.widgets.get("sending_facility")
receiving_app = dbutils.widgets.get("receiving_app")
receiving_facility = dbutils.widgets.get("receiving_facility")
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
volume_name = dbutils.widgets.get("volume_name")
relative_path = dbutils.widgets.get("relative_path")

# Volume path: /Volumes/<catalog>/<schema>/<volume>/<path_to_file>
output_path = f"/Volumes/{catalog_use}/{schema_use}/{volume_name}/{relative_path}"

In [0]:
volume_details = spark.sql(f"""
  SELECT
    v.volume_catalog,
    v.volume_schema,
    v.volume_name,
    v.storage_location,
    e.external_location_name,
    e.url AS external_location_url
  FROM system.information_schema.volumes v
  JOIN system.information_schema.external_locations e
    -- storage_location is a subpath under the external location URL
    ON lower(v.storage_location) LIKE concat(lower(e.url), '%')
  WHERE v.volume_catalog = '{catalog_use}'
    AND v.volume_schema  = '{schema_use}'
    AND v.volume_name    = '{volume_name}'
  ORDER BY length(e.url) DESC   -- pick the most specific matching location
  LIMIT 1
""")

In [0]:
storage_location_str = volume_details.collect()[0]["storage_location"]
storage_location_str

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from datetime import datetime

# ============================================================================
# SEGMENT-SPECIFIC UDFs - Each handles one HL7 segment
# ============================================================================

# --- MSH Segment UDF ---
@udf(returnType=StringType())
def build_msh_segment(msg_datetime, msg_control_id, event_type, sending_app, sending_facility, receiving_app, receiving_facility, hl7_version):
    """Build MSH (Message Header) segment"""
    return f"MSH|^~\\&|{sending_app}|{sending_facility}|{receiving_app}|{receiving_facility}|{msg_datetime}||{event_type}|{msg_control_id}|P|{hl7_version}||"

# --- EVN Segment UDF ---
@udf(returnType=StringType())
def build_evn_segment(event_type, event_datetime):
    """Build EVN (Event Type) segment"""
    return f"EVN|{event_type}|{event_datetime}||"

# --- PID Segment UDF ---
@udf(returnType=StringType())
def build_pid_segment(mrn, ssn, last_name, first_name, middle_initial, birth_date, sex, 
                      race, marital_status, religion, ethnic_group, birth_place,
                      addr_street, addr_city, addr_state, addr_zip, phone_home):
    """Build PID (Patient Identification) segment with Phase 1 & 2 enhancements"""
    return (f"PID|1||{mrn}^^^MRN||{last_name}^{first_name}^{middle_initial}||{birth_date}|{sex}||{race}|" +
            f"{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA||{phone_home}|||{marital_status}|" +
            f"{religion}||||{ssn}|||{ethnic_group}|{birth_place}||")

# --- NK1 Segment UDF ---
@udf(returnType=StringType())
def build_nk1_segment(nok_last_name, nok_first_name, nok_relationship, nok_phone):
    """Build NK1 (Next of Kin) segment"""
    return f"NK1|1|{nok_last_name}^{nok_first_name}|{nok_relationship}|{nok_phone}||"

# --- PV1 Segment UDF ---
@udf(returnType=StringType())
def build_pv1_segment(patient_class, patient_location, attending_dr_id, attending_dr_last, attending_dr_first,
                      hospital_service, admit_datetime, discharge_datetime, event_type):
    """Build PV1 (Patient Visit) segment with Phase 1 enhancements"""
    # PV1-45 (discharge datetime) only for A03 events
    discharge_dt = discharge_datetime if event_type == 'A03' else ''
    return (f"PV1|1|{patient_class}|{patient_location}|||{attending_dr_id}^{attending_dr_last}^{attending_dr_first}^^^^MD|" +
            f"|||{hospital_service}|||||||||{attending_dr_id}^{attending_dr_last}^{attending_dr_first}^^^^MD|" +
            f"||||||||||||||||||||||||||||||||{admit_datetime}|{discharge_dt}||")

# --- PV2 Segment UDF ---
@udf(returnType=StringType())
def build_pv2_segment(admit_reason, expected_discharge):
    """Build PV2 (Patient Visit Additional) segment - Phase 2 enhancement"""
    return f"PV2|||{admit_reason}|||||{expected_discharge}|"

# --- DG1 Segment UDF ---
@udf(returnType=StringType())
def build_dg1_segment(diagnosis_code, diagnosis_desc, diagnosis_type, msg_datetime):
    """Build DG1 (Diagnosis) segment - Phase 1 enhancement"""
    return f"DG1|1||{diagnosis_code}^{diagnosis_desc}||{msg_datetime}|{diagnosis_type}||"

# --- AL1 Segment UDF (Conditional) ---
@udf(returnType=StringType())
def build_al1_segments(has_allergies, 
                       allergy1_type, allergy1_code_desc, allergy1_severity, allergy1_reaction,
                       has_allergy2,
                       allergy2_type, allergy2_code_desc, allergy2_severity, allergy2_reaction):
    """Build AL1 (Allergy) segments - Phase 2 enhancement. Returns 0-2 segments."""
    if has_allergies != 'Y':
        return ''  # No allergies
    
    segments = []
    
    # First allergy
    segments.append(f"AL1|1|{allergy1_type}|{allergy1_code_desc}|{allergy1_severity}|{allergy1_reaction}|")
    
    # Second allergy (if exists)
    if has_allergy2 == 'Y':
        segments.append(f"AL1|2|{allergy2_type}|{allergy2_code_desc}|{allergy2_severity}|{allergy2_reaction}|")
    
    return '\r'.join(segments)

# --- GT1 Segment UDF ---
@udf(returnType=StringType())
def build_gt1_segment(last_name, first_name, middle_initial, addr_street, addr_city, addr_state, addr_zip, phone_home, birth_date, sex):
    """Build GT1 (Guarantor) segment"""
    return (f"GT1|1||{last_name}^{first_name}^{middle_initial}||{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA|" +
            f"{phone_home}|||{birth_date}|{sex}||01||")

# --- IN1 Segment UDF ---
@udf(returnType=StringType())
def build_in1_segment(ins_plan_id, ins_co_name, ins_co_addr_street, ins_co_addr_city, ins_co_addr_state, ins_co_addr_zip,
                      ins_co_phone, ins_group_num, ins_group_name, ins_policy_num, 
                      last_name, first_name, middle_initial, birth_date, 
                      addr_street, addr_city, addr_state, addr_zip, ins_relationship, ins_effective_date):
    """Build IN1 (Insurance) segment"""
    return (f"IN1|1|{ins_plan_id}|{ins_plan_id}|{ins_co_name}|{ins_co_addr_street}^^{ins_co_addr_city}^{ins_co_addr_state}^{ins_co_addr_zip}^USA|" +
            f"|{ins_co_phone}|{ins_group_num}||||{ins_effective_date}||||{last_name}^{first_name}^{middle_initial}|" +
            f"{ins_relationship}|{birth_date}|{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA|||||||||||||||||" +
            f"{ins_policy_num}||||||||||{ins_group_name}||")

# --- IN2 Segment UDF ---
@udf(returnType=StringType())
def build_in2_segment(ssn, ins_expiration_date):
    """Build IN2 (Insurance Additional Information) segment"""
    return f"IN2||{ssn}|||||||||||||||||||||||||||||||||||||||||||||||||||{ins_expiration_date}|"

print("✅ Created 11 segment-specific UDFs:")
print("   • MSH (Message Header)")
print("   • EVN (Event Type)")
print("   • PID (Patient Identification - 17 params)")
print("   • NK1 (Next of Kin)")
print("   • PV1 (Patient Visit - 9 params)")
print("   • PV2 (Patient Visit Additional - 2 params)")
print("   • DG1 (Diagnosis - 4 params)")
print("   • AL1 (Allergy - conditional 0-2 segments, 9 params)")
print("   • GT1 (Guarantor)")
print("   • IN1 (Insurance - 20 params)")
print("   • IN2 (Insurance Additional - 2 params)")

In [0]:
# -------------------------------------------------------------------
# 1) Generate base patient dimension (synthetic with Faker)
# -------------------------------------------------------------------
from faker import Faker
import random
import string
from datetime import datetime, timedelta

# UDFs to generate realistic patient data using Faker
@F.udf("string")
def generate_first_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.first_name()

@F.udf("string")
def generate_last_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.last_name()

@F.udf("string")
def generate_dob(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    # Generate DOB for adults aged 18-90
    dob = fake.date_of_birth(minimum_age=18, maximum_age=90)
    return dob.strftime("%Y%m%d")

@F.udf("string")
def generate_patient_id(seed_val, first_name, last_name):
    # Create patient ID: first 2 letters of last name + first letter of first name + 6 digits
    prefix = (last_name[:2] + first_name[:1]).upper()
    # Use seed to generate consistent number
    random.seed(seed_val)
    number = str(random.randint(100000, 999999))
    return f"{prefix}{number}"

@F.udf("string")
def generate_visit_id(seed_val):
    # Generate realistic visit ID: VIS + YYMMDD + 4-character alphanumeric
    fake = Faker()
    Faker.seed(seed_val)
    random.seed(seed_val)
    
    # Get a date component (using current year)
    visit_date = fake.date_between(start_date='-30d', end_date='today')
    date_str = visit_date.strftime("%y%m%d")
    
    # Generate 4-character alphanumeric suffix
    chars = string.ascii_uppercase + string.digits
    suffix = ''.join(random.choice(chars) for _ in range(4))
    
    return f"VIS{date_str}{suffix}"

@F.udf("string")
def generate_ssn(seed_val):
    # Generate fake SSN: ###-##-####
    random.seed(seed_val)
    area = str(random.randint(100, 899))  # Avoid certain ranges
    group = str(random.randint(10, 99))
    serial = str(random.randint(1000, 9999))
    return f"{area}-{group}-{serial}"

@F.udf("string")
def generate_street_address(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.street_address()

@F.udf("string")
def generate_city(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.city()

@F.udf("string")
def generate_state(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.state_abbr()

@F.udf("string")
def generate_zipcode(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.zipcode()

@F.udf("string")
def generate_phone(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    # Format: ###-###-####
    phone = fake.phone_number()
    # Clean up to basic format
    digits = ''.join(filter(str.isdigit, phone))[:10]
    if len(digits) == 10:
        return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"
    return phone

# PHASE 1 & 2: Additional demographic UDFs
@F.udf("string")
def generate_race(seed_val):
    # HL7 race codes from CDC Race & Ethnicity code system
    random.seed(seed_val)
    races = [
        ("2106-3", 60),  # White - 60%
        ("2054-5", 20),  # Black or African American - 20%
        ("2028-9", 10),  # Asian - 10%
        ("2076-8", 5),   # Native Hawaiian or Other Pacific Islander - 5%
        ("1002-5", 5),   # American Indian or Alaska Native - 5%
    ]
    # Weighted random selection
    roll = random.randint(1, 100)
    cumulative = 0
    for code, weight in races:
        cumulative += weight
        if roll <= cumulative:
            return code
    return "2106-3"  # Default to White

@F.udf("string")
def generate_marital_status(seed_val):
    # HL7 marital status codes
    random.seed(seed_val)
    statuses = ["M", "S", "D", "W", "A"]  # Married, Single, Divorced, Widowed, Separated
    weights = [45, 30, 15, 7, 3]  # Weighted distribution
    roll = random.randint(1, 100)
    cumulative = 0
    for status, weight in zip(statuses, weights):
        cumulative += weight
        if roll <= cumulative:
            return status
    return "M"

@F.udf("string")
def generate_religion(seed_val):
    # HL7 religion codes
    random.seed(seed_val)
    religions = ["CAT", "BAP", "PRO", "JEW", "MOS", "OTH", "NON"]
    weights = [25, 15, 20, 5, 3, 12, 20]  # Weighted distribution
    roll = random.randint(1, 100)
    cumulative = 0
    for religion, weight in zip(religions, weights):
        cumulative += weight
        if roll <= cumulative:
            return religion
    return "OTH"

@F.udf("string")
def generate_ethnic_group(seed_val):
    # Hispanic/Latino ethnicity
    random.seed(seed_val)
    return "H" if random.randint(1, 100) <= 18 else "N"  # ~18% Hispanic

@F.udf("string")
def generate_birth_place(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    city = fake.city()
    state = fake.state_abbr()
    return f"{city}, {state}"

# PHASE 1: Diagnosis data
@F.udf("string")
def generate_diagnosis_code(seed_val):
    # Common ICD-10 codes for realistic hospital admissions
    random.seed(seed_val)
    diagnoses = [
        "E11.9",    # Type 2 diabetes mellitus without complications
        "I10",      # Essential (primary) hypertension
        "J44.9",    # COPD, unspecified
        "I25.10",   # Atherosclerotic heart disease
        "N18.3",    # Chronic kidney disease, stage 3
        "J18.9",    # Pneumonia, unspecified organism
        "I50.9",    # Heart failure, unspecified
        "K21.9",    # Gastro-esophageal reflux disease
        "M25.50",   # Pain in unspecified joint
        "E78.5",    # Hyperlipidemia, unspecified
        "F32.9",    # Major depressive disorder, single episode
        "M54.5",    # Low back pain
        "I63.9",    # Cerebral infarction, unspecified
        "J96.00",   # Acute respiratory failure
        "N39.0",    # Urinary tract infection
    ]
    return random.choice(diagnoses)

@F.udf("string")
def generate_diagnosis_desc(diagnosis_code):
    # Map ICD-10 to description
    desc_map = {
        "E11.9": "Type 2 diabetes mellitus without complications",
        "I10": "Essential hypertension",
        "J44.9": "COPD, unspecified",
        "I25.10": "Atherosclerotic heart disease",
        "N18.3": "Chronic kidney disease, stage 3",
        "J18.9": "Pneumonia, unspecified",
        "I50.9": "Heart failure, unspecified",
        "K21.9": "Gastro-esophageal reflux disease",
        "M25.50": "Pain in unspecified joint",
        "E78.5": "Hyperlipidemia, unspecified",
        "F32.9": "Major depressive disorder",
        "M54.5": "Low back pain",
        "I63.9": "Cerebral infarction, unspecified",
        "J96.00": "Acute respiratory failure",
        "N39.0": "Urinary tract infection",
    }
    return desc_map.get(diagnosis_code, "Unspecified diagnosis")

@F.udf("string")
def generate_diagnosis_type(seed_val):
    # A=Admitting, W=Working, F=Final
    random.seed(seed_val)
    return random.choice(["A", "W", "F"])

@F.udf("string")
def generate_hospital_service(seed_val):
    # Hospital service codes
    random.seed(seed_val)
    services = ["MED", "SUR", "OBS", "CAR", "NEU", "ONC", "ORT", "PSY", "PED"]
    weights = [30, 20, 15, 10, 8, 7, 5, 3, 2]  # Weighted distribution
    roll = random.randint(1, 100)
    cumulative = 0
    for service, weight in zip(services, weights):
        cumulative += weight
        if roll <= cumulative:
            return service
    return "MED"

# PHASE 2: Allergy data (30% of patients have allergies)
@F.udf("string")
def generate_has_allergies(seed_val):
    random.seed(seed_val)
    return "Y" if random.randint(1, 100) <= 30 else "N"

@F.udf("string")
def generate_allergy_type(seed_val):
    random.seed(seed_val)
    return random.choice(["DA", "FA", "MA", "EA"])  # Drug, Food, Misc, Environmental

@F.udf("string")
def generate_allergy_code_desc(seed_val, allergy_type):
    # Return code^description
    random.seed(seed_val)
    drug_allergies = [
        "1191^Aspirin",
        "7980^Penicillin",
        "203^Ibuprofen",
        "1605^Codeine",
        "1272^Morphine",
    ]
    food_allergies = [
        "F01^Peanuts",
        "F02^Shellfish",
        "F03^Eggs",
        "F04^Milk",
        "F05^Soy",
    ]
    misc_allergies = [
        "M01^Latex",
        "M02^Contrast dye",
        "M03^Adhesive tape",
    ]
    env_allergies = [
        "E01^Pollen",
        "E02^Dust mites",
        "E03^Pet dander",
    ]
    
    if allergy_type == "DA":
        return random.choice(drug_allergies)
    elif allergy_type == "FA":
        return random.choice(food_allergies)
    elif allergy_type == "MA":
        return random.choice(misc_allergies)
    else:
        return random.choice(env_allergies)

@F.udf("string")
def generate_allergy_severity(seed_val):
    random.seed(seed_val)
    return random.choice(["MI", "MO", "SV"])  # Mild, Moderate, Severe

@F.udf("string")
def generate_allergy_reaction(seed_val):
    random.seed(seed_val)
    reactions = ["Rash", "Hives", "Itching", "Swelling", "Nausea", "Anaphylaxis", "Shortness of breath"]
    # Pick 1-2 reactions
    n_reactions = random.choice([1, 2])
    selected = random.sample(reactions, n_reactions)
    return "^".join(selected)

# PHASE 2: PV2 data
@F.udf("string")
def generate_admit_reason(seed_val):
    random.seed(seed_val)
    reasons = [
        "Chest pain",
        "Shortness of breath",
        "Abdominal pain",
        "Scheduled surgery",
        "Fall with injury",
        "Fever and weakness",
        "Altered mental status",
        "Syncope",
        "Seizure",
        "Motor vehicle accident",
    ]
    return random.choice(reasons)

@F.udf("string")
def generate_insurance_effective_date(seed_val):
    # Events occur in January 2026, so generate effective dates 6-18 months before
    # This ensures recent, realistic coverage that's valid during events
    event_base = datetime(2026, 1, 1)
    random.seed(seed_val)
    days_before = random.randint(180, 540)  # 6-18 months before events
    effective_date = event_base - timedelta(days=days_before)
    return effective_date.strftime("%Y%m%d")

@F.udf("string")
def generate_insurance_expiration_date(seed_val, effective_date_str):
    # Generate expiration that ensures coverage through at least mid-2026
    # Set expiration 18-30 months after effective date
    try:
        effective_date = datetime.strptime(effective_date_str, "%Y%m%d")
        event_base = datetime(2026, 1, 1)
        random.seed(seed_val)
        
        # Calculate minimum months needed to cover events (from effective to mid-2026)
        months_to_events = ((event_base.year - effective_date.year) * 12 + 
                           (event_base.month - effective_date.month))
        # Add 6-12 months buffer beyond events
        min_coverage_months = months_to_events + 6
        max_coverage_months = months_to_events + 18
        
        # Generate expiration date
        coverage_days = random.randint(min_coverage_months * 30, max_coverage_months * 30)
        expiration_date = effective_date + timedelta(days=coverage_days)
        
        return expiration_date.strftime("%Y%m%d")
    except:
        return ""

patients_df = (
    spark.range(0, num_patients)
    .withColumnRenamed("id", "patient_index")
    .withColumn("seed", (F.col("patient_index") + 12345).cast("int"))  # Add offset for variety
    .withColumn("first_name", generate_first_name(F.col("seed")))
    .withColumn("last_name", generate_last_name(F.col("seed")))
    .withColumn("dob", generate_dob(F.col("seed")))
    .withColumn("patient_id", generate_patient_id(F.col("seed"), F.col("first_name"), F.col("last_name")))
    .withColumn("visit_id", generate_visit_id(F.col("seed")))
    .withColumn("sex", F.when((F.col("patient_index") % 2) == 0, "M").otherwise("F"))
    .withColumn("ssn", generate_ssn(F.col("seed") + 5))
    # Address fields
    .withColumn("street_address", generate_street_address(F.col("seed")))
    .withColumn("city", generate_city(F.col("seed") + 1))
    .withColumn("state", generate_state(F.col("seed") + 2))
    .withColumn("zipcode", generate_zipcode(F.col("seed") + 3))
    .withColumn("country", F.lit("USA"))
    .withColumn("phone", generate_phone(F.col("seed") + 4))
    # PHASE 1 & 2: Additional demographics
    .withColumn("race", generate_race(F.col("seed") + 100))
    .withColumn("marital_status", generate_marital_status(F.col("seed") + 101))
    .withColumn("religion", generate_religion(F.col("seed") + 102))
    .withColumn("ethnic_group", generate_ethnic_group(F.col("seed") + 103))
    .withColumn("birth_place", generate_birth_place(F.col("seed") + 104))
    # Next of kin fields (using offset seed for different person)
    .withColumn("nok_first_name", generate_first_name(F.col("seed") + 1000))
    .withColumn("nok_last_name", generate_last_name(F.col("seed") + 1001))
    .withColumn("nok_phone", generate_phone(F.col("seed") + 1002))
    .withColumn("nok_relationship", F.when((F.col("patient_index") % 3) == 0, "SPO").when((F.col("patient_index") % 3) == 1, "CHD").otherwise("SIB"))  # Spouse, Child, Sibling
    # PHASE 1: Diagnosis data
    .withColumn("diagnosis_code", generate_diagnosis_code(F.col("seed") + 200))
    .withColumn("diagnosis_desc", generate_diagnosis_desc(F.col("diagnosis_code")))
    .withColumn("diagnosis_type", generate_diagnosis_type(F.col("seed") + 201))
    # PHASE 1: Hospital service
    .withColumn("hospital_service", generate_hospital_service(F.col("seed") + 300))
    # PHASE 2: Allergy data (30% of patients)
    .withColumn("has_allergies", generate_has_allergies(F.col("seed") + 400))
    # Allergy 1
    .withColumn("allergy1_type", F.when(F.col("has_allergies") == "Y", generate_allergy_type(F.col("seed") + 401)).otherwise(""))
    .withColumn("allergy1_code_desc", F.when(F.col("has_allergies") == "Y", generate_allergy_code_desc(F.col("seed") + 402, F.col("allergy1_type"))).otherwise(""))
    .withColumn("allergy1_severity", F.when(F.col("has_allergies") == "Y", generate_allergy_severity(F.col("seed") + 403)).otherwise(""))
    .withColumn("allergy1_reaction", F.when(F.col("has_allergies") == "Y", generate_allergy_reaction(F.col("seed") + 404)).otherwise(""))
    # Allergy 2 (50% of patients with allergies have a second allergy)
    .withColumn("has_allergy2", F.when((F.col("has_allergies") == "Y") & ((F.col("seed") % 2) == 0), "Y").otherwise("N"))
    .withColumn("allergy2_type", F.when(F.col("has_allergy2") == "Y", generate_allergy_type(F.col("seed") + 405)).otherwise(""))
    .withColumn("allergy2_code_desc", F.when(F.col("has_allergy2") == "Y", generate_allergy_code_desc(F.col("seed") + 406, F.col("allergy2_type"))).otherwise(""))
    .withColumn("allergy2_severity", F.when(F.col("has_allergy2") == "Y", generate_allergy_severity(F.col("seed") + 407)).otherwise(""))
    .withColumn("allergy2_reaction", F.when(F.col("has_allergy2") == "Y", generate_allergy_reaction(F.col("seed") + 408)).otherwise(""))
    # PHASE 2: PV2 data
    .withColumn("admit_reason", generate_admit_reason(F.col("seed") + 500))
    # Insurance fields
    .withColumn("insurance_company", F.when((F.col("patient_index") % 4) == 0, "BLUE CROSS").when((F.col("patient_index") % 4) == 1, "AETNA").when((F.col("patient_index") % 4) == 2, "CIGNA").otherwise("UNITED HEALTHCARE"))
    .withColumn("insurance_company_id", F.when((F.col("patient_index") % 4) == 0, "BCBS").when((F.col("patient_index") % 4) == 1, "AETN").when((F.col("patient_index") % 4) == 2, "CIGN").otherwise("UNHC"))
    .withColumn("insurance_id", F.concat(F.lit("INS"), F.lpad((F.col("patient_index") + 10000).cast("string"), 9, "0")))
    .withColumn("insurance_group", F.concat(F.lit("GRP"), F.lpad((F.col("patient_index") % 100).cast("string"), 5, "0")))
    .withColumn("insurance_plan_id", F.concat(F.lit("PLAN"), F.lpad((F.col("patient_index") % 50).cast("string"), 4, "0")))
    .withColumn("insurance_plan_type", F.when((F.col("patient_index") % 3) == 0, "HMO").when((F.col("patient_index") % 3) == 1, "PPO").otherwise("POS"))
    .withColumn("insurance_effective_date", generate_insurance_effective_date(F.col("seed") + 6))
    .withColumn("insurance_expiration_date", generate_insurance_expiration_date(F.col("seed") + 7, F.col("insurance_effective_date")))
    .drop("seed")
)

In [0]:
display(patients_df, 10)

In [0]:
# -------------------------------------------------------------------
# 2) Event plan per patient with realistic event types
# -------------------------------------------------------------------
event_schema = T.ArrayType(
    T.StructType([
        T.StructField("event_order", T.IntegerType(), False),
        T.StructField("event_type", T.StringType(), False)
    ])
)

@F.udf(event_schema)
def build_event_plan(random_int):
    events = []
    order = 0
    
    # Determine patient journey type based on random value
    journey_type = random_int % 10
    
    if journey_type == 0:
        # Journey 1: ER registration -> Admit -> Discharge
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        events.append({"event_order": order, "event_type": "A06"})  # Convert to inpatient
        order += 1
        # Add some updates
        for _ in range((random_int // 10) % 3):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 1:
        # Journey 2: Direct admit -> Transfer -> Discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        # Add transfers
        for _ in range((random_int // 20) % 2 + 1):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 2:
        # Journey 3: Admit -> Cancel admit (patient left/error)
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A11"})  # Cancel admit
        
    elif journey_type == 3:
        # Journey 4: Admit -> Transfer planned -> Cancel transfer -> Eventually transfer -> Discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A02"})
        order += 1
        events.append({"event_order": order, "event_type": "A12"})  # Cancel transfer
        order += 1
        events.append({"event_order": order, "event_type": "A02"})  # Actual transfer
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 4:
        # Journey 5: Admit -> Discharge planned -> Cancel discharge -> Eventually discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        order += 1
        events.append({"event_order": order, "event_type": "A13"})  # Cancel discharge
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})  # Actual discharge
        
    elif journey_type == 5:
        # Journey 6: ER registration only (no admission)
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        # Maybe some updates
        if (random_int // 50) % 2 == 0:
            events.append({"event_order": order, "event_type": "A08"})
            
    elif journey_type == 6:
        # Journey 7: Inpatient to outpatient status change
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A07"})  # Change to outpatient
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    else:
        # Journey 8-9: Standard admit with updates and discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        
        # Add varying number of updates
        n_updates = (random_int % 4)
        for _ in range(n_updates):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
            
        # Maybe add transfers
        n_transfers = (random_int // 4) % 3
        for _ in range(n_transfers):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
            
        # Most patients get discharged
        if (random_int % 5 != 0):
            events.append({"event_order": order, "event_type": "A03"})

    return events

events_planned = (
    patients_df
    .withColumn("rand", (F.rand() * 1000).cast("int"))
    .withColumn("events", build_event_plan(F.col("rand")))
    .drop("rand")  # Keep all patient fields
)

events_exploded = (
    events_planned
    .withColumn("event", F.explode("events"))
    .withColumn("event_order", F.col("event.event_order"))
    .withColumn("event_type", F.col("event.event_type"))
    .drop("event", "events")  # Drop the array columns, keep all 46 patient fields + event fields
)

print(f"✅ events_exploded created with all {len(events_exploded.columns)} columns from patient dimension plus event fields")
print(f"   Row count: {events_exploded.count()}")
print(f"   Sample event types: {[row.event_type for row in events_exploded.select('event_type').distinct().limit(5).collect()]}")

In [0]:
# -------------------------------------------------------------------
# 3) Add timing and visit context
# -------------------------------------------------------------------
base_ts = F.to_timestamp(F.lit("2026-01-01 00:00:00"), "yyyy-MM-dd HH:mm:ss")

events_timed = (
    events_exploded
    .withColumn("hours_offset", (F.col("event_order") * 2 + (F.hash(F.col("patient_id")) % 24)).cast("int"))
    .withColumn(
        "event_datetime",
        F.date_format(
            base_ts + F.expr("MAKE_INTERVAL(0, 0, 0, 0, hours_offset, 0, 0)"),
            "yyyyMMddHHmmss"
        )
    )
    # Add message control ID (unique identifier for each message)
    .withColumn("msg_control_id", F.concat(F.lit("MSG"), F.monotonically_increasing_id().cast("string")))
    # Add msg_datetime as alias for event_datetime (HL7 convention)
    .withColumn("msg_datetime", F.col("event_datetime"))
    .withColumn(
        "patient_class",
        F.when(F.col("event_type") == "A04", "E")  # Emergency/ER registration
         .when(F.col("event_type") == "A06", "I")  # Converting to inpatient
         .when(F.col("event_type") == "A07", "O")  # Converting to outpatient
         .when(F.col("event_type").isin("A01", "A02", "A03"), "I")  # Inpatient events
         .when(F.col("event_type").isin("A11", "A12", "A13"), "I")  # Cancellations
         .otherwise("O")  # Updates default to outpatient
    )
    .withColumn(
        "patient_location",  # Renamed from 'location' to match HL7 UDF parameter
        F.concat(
            F.when(F.col("event_type") == "A04", F.lit("ER"))
             .otherwise(F.lit("WARD")),
            F.lpad((F.hash(F.col("patient_id")) % 10).cast("int").cast("string"), 2, "0"),
            F.lit("-ROOM"),
            F.lpad((F.hash(F.col("visit_id")) % 20).cast("int").cast("string"), 2, "0")
        )
    )
    .withColumn("attending_dr_id", F.concat(F.lit("DR"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_dr_last", F.concat(F.lit("DOC"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_dr_first", F.lit("ATTENDING"))
    .withColumn(
        "discharge_datetime",  # Renamed from 'discharge_disposition' - will be used for PV1-45
        F.when(F.col("event_type") == "A03", F.col("event_datetime")).otherwise("")
    )
    # Calculate admit_datetime (first A01 or A04 event per patient)
    .withColumn(
        "admit_datetime",
        F.first(
            F.when(F.col("event_type").isin("A01", "A04"), F.col("event_datetime")).otherwise(None)
        ).over(Window.partitionBy("patient_id").orderBy("event_order").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
    )
    # Calculate expected discharge (admit + 3-7 days)
    .withColumn(
        "expected_discharge_ts",
        F.when(
            F.col("admit_datetime").isNotNull(),
            F.to_timestamp(F.col("admit_datetime"), "yyyyMMddHHmmss") + F.expr("MAKE_INTERVAL(0, 0, 0, 3 + (abs(hash(patient_id)) % 5), 0, 0, 0)")
        )
    )
    .withColumn(
        "expected_discharge",
        F.when(F.col("expected_discharge_ts").isNotNull(), F.date_format(F.col("expected_discharge_ts"), "yyyyMMddHHmmss")).otherwise("")
    )
    .drop("hours_offset", "expected_discharge_ts")
)

from pyspark.sql import Window
print(f"✅ events_timed created with timing and visit context")
print(f"   Row count: {events_timed.count()}")
print(f"   Columns: {len(events_timed.columns)}")
print(f"\nNew columns added:")
print(f"   • msg_control_id: Unique message identifier")
print(f"   • msg_datetime: Message timestamp (same as event_datetime)")
print(f"   • admit_datetime: First admit event timestamp per patient")
print(f"   • expected_discharge: Calculated expected discharge date")

In [0]:
display(events_timed)

In [0]:
from pyspark.sql import functions as F

# HL7 Configuration (as literals)
hl7_config = {
    'sending_app': 'SyntheticEHR',
    'sending_facility': 'TestHospital',
    'receiving_app': 'DataPlatform',
    'receiving_facility': 'Analytics',
    'hl7_version': '2.5'
}

print("Building HL7 messages using segment-specific UDFs...")

# Step 1: Apply each segment UDF to create individual segment columns
with_segments = events_timed \
    .withColumn('seg_msh', build_msh_segment(
        F.col('msg_datetime'), F.col('msg_control_id'), F.col('event_type'),
        F.lit(hl7_config['sending_app']), F.lit(hl7_config['sending_facility']),
        F.lit(hl7_config['receiving_app']), F.lit(hl7_config['receiving_facility']),
        F.lit(hl7_config['hl7_version'])
    )) \
    .withColumn('seg_evn', build_evn_segment(
        F.col('event_type'), F.col('event_datetime')
    )) \
    .withColumn('seg_pid', build_pid_segment(
        F.col('patient_id'),  # mrn
        F.col('ssn'), 
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),  # middle_initial - not in source data
        F.col('dob'),  # birth_date
        F.col('sex'), 
        F.col('race'), 
        F.col('marital_status'), 
        F.col('religion'), 
        F.col('ethnic_group'), 
        F.col('birth_place'),
        F.col('street_address'),  # addr_street
        F.col('city'),  # addr_city
        F.col('state'),  # addr_state
        F.col('zipcode'),  # addr_zip
        F.col('phone')  # phone_home
    )) \
    .withColumn('seg_nk1', build_nk1_segment(
        F.col('nok_last_name'), F.col('nok_first_name'), F.col('nok_relationship'), F.col('nok_phone')
    )) \
    .withColumn('seg_pv1', build_pv1_segment(
        F.col('patient_class'), 
        F.col('patient_location'),
        F.col('attending_dr_id'), 
        F.col('attending_dr_last'), 
        F.col('attending_dr_first'),
        F.col('hospital_service'), 
        F.col('admit_datetime'), 
        F.col('discharge_datetime'), 
        F.col('event_type')
    )) \
    .withColumn('seg_pv2', build_pv2_segment(
        F.col('admit_reason'), F.col('expected_discharge')
    )) \
    .withColumn('seg_dg1', build_dg1_segment(
        F.col('diagnosis_code'), F.col('diagnosis_desc'), F.col('diagnosis_type'), F.col('msg_datetime')
    )) \
    .withColumn('seg_al1', build_al1_segments(
        F.col('has_allergies'),
        F.col('allergy1_type'), F.col('allergy1_code_desc'), F.col('allergy1_severity'), F.col('allergy1_reaction'),
        F.col('has_allergy2'),
        F.col('allergy2_type'), F.col('allergy2_code_desc'), F.col('allergy2_severity'), F.col('allergy2_reaction')
    )) \
    .withColumn('seg_gt1', build_gt1_segment(
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),  # middle_initial
        F.col('street_address'),  # addr_street
        F.col('city'),  # addr_city
        F.col('state'),  # addr_state
        F.col('zipcode'),  # addr_zip
        F.col('phone'),  # phone_home
        F.col('dob'),  # birth_date
        F.col('sex')
    )) \
    .withColumn('seg_in1', build_in1_segment(
        F.col('insurance_plan_id'), 
        F.col('insurance_company'),  # ins_co_name
        F.lit(''),  # ins_co_addr_street - not in source
        F.lit(''),  # ins_co_addr_city - not in source
        F.lit(''),  # ins_co_addr_state - not in source
        F.lit(''),  # ins_co_addr_zip - not in source
        F.lit(''),  # ins_co_phone - not in source
        F.col('insurance_group'),  # ins_group_num
        F.lit(''),  # ins_group_name - not in source
        F.col('insurance_id'),  # ins_policy_num
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),  # middle_initial
        F.col('dob'),  # birth_date
        F.col('street_address'),  # addr_street
        F.col('city'),  # addr_city
        F.col('state'),  # addr_state
        F.col('zipcode'),  # addr_zip
        F.lit('18'),  # ins_relationship - default to 'self'
        F.col('insurance_effective_date')
    )) \
    .withColumn('seg_in2', build_in2_segment(
        F.col('ssn'), F.col('insurance_expiration_date')
    ))

print(f"  • Created 11 segment columns: seg_msh, seg_evn, seg_pid, seg_nk1, seg_pv1, seg_pv2, seg_dg1, seg_al1, seg_gt1, seg_in1, seg_in2")

# Step 2: Concatenate segments with carriage return separators
# Handle AL1 segments specially (can be empty, 1 segment, or 2 segments)
hl7_messages = with_segments.withColumn(
    'hl7_message',
    F.concat_ws(
        '\r',  # HL7 segment separator
        F.col('seg_msh'),
        F.col('seg_evn'),
        F.col('seg_pid'),
        F.col('seg_nk1'),
        F.col('seg_pv1'),
        F.col('seg_pv2'),
        F.col('seg_dg1'),
        # AL1 segments (already contains '\r' separator if 2 allergies, empty string if none)
        F.when(F.col('seg_al1') != '', F.col('seg_al1')).otherwise(F.lit(None)),
        F.col('seg_gt1'),
        F.col('seg_in1'),
        F.col('seg_in2')
    )
)

print(f"\n✅ HL7 messages constructed using multi-UDF approach")
print(f"   Total events: {hl7_messages.count()}")
print(f"\nSegment order: MSH → EVN → PID → NK1 → PV1 → PV2 → DG1 → AL1(0-2) → GT1 → IN1 → IN2")
print(f"\nNote: Serverless compute doesn't support caching. Operations may recompute UDFs.")

In [0]:
# Show statistics without materializing all data
print("="*80)
print("HL7 MESSAGE GENERATION SUMMARY - PHASE 1 & 2 ENHANCEMENTS")
print("="*80)

print(f"\n📄 Total HL7 Messages Generated: {hl7_messages.count()}")

# Event type distribution
print(f"\n📊 Event Type Distribution:")
event_stats = hl7_messages.groupBy('event_type').count().orderBy('event_type')
event_stats.show(20, truncate=False)

# Allergy statistics
print(f"\n🤧 Allergy Statistics (Phase 2):")
allergy_stats = hl7_messages.select(
    F.sum(F.when(F.col('has_allergies') == 'Y', 1).otherwise(0)).alias('with_allergies'),
    F.sum(F.when(F.col('has_allergies') == 'N', 1).otherwise(0)).alias('no_allergies'),
    F.sum(F.when(F.col('has_allergy2') == 'Y', 1).otherwise(0)).alias('with_2_allergies')
).collect()[0]

print(f"   • Messages with allergies (AL1 segments): {allergy_stats.with_allergies}")
print(f"   • Messages without allergies: {allergy_stats.no_allergies}")
print(f"   • Messages with 2 allergies: {allergy_stats.with_2_allergies}")

# Phase 1 & 2 field coverage
print(f"\n✔️ Phase 1 Enhancements Included:")
print(f"   • DG1 segment (Diagnosis with ICD-10 codes)")
print(f"   • PID-10 (race), PID-16 (marital status)")
print(f"   • PV1-10 (hospital service), PV1-44/45 (admit/discharge dates)")

print(f"\n✔️ Phase 2 Enhancements Included:")
print(f"   • PID-17 (religion), PID-22 (ethnic group), PID-23 (birth place)")
print(f"   • AL1 segment(s) - Allergies (0-2 per patient)")
print(f"   • PV2 segment - Admit reason, expected discharge")

print(f"\n📦 Segment Structure (11 segment types):")
print(f"   MSH → EVN → PID → NK1 → PV1 → PV2 → DG1 → AL1(0-2) → GT1 → IN1 → IN2")

print(f"\n✅ Multi-UDF Architecture: 11 smaller UDFs (4-20 params each)")
print(f"   vs. previous monolithic UDF (63 params) - Avoids Python worker crashes")
print("="*80)

In [0]:
# Display a sample HL7 message (with allergies to show all segments)
print("="*80)
print("SAMPLE HL7 ADT MESSAGE (Patient with 2 Allergies)")
print("="*80)

sample_row = hl7_messages.filter(
    (F.col('has_allergies') == 'Y') & 
    (F.col('has_allergy2') == 'Y')
).select('patient_id', 'event_type', 'hl7_message').first()

print(f"\nPatient ID: {sample_row.patient_id}")
print(f"Event Type: {sample_row.event_type}")
print(f"\nHL7 Message Segments:\n{'-'*80}")

# Split message by carriage return and display each segment
segments = sample_row.hl7_message.split('\r')
for i, segment in enumerate(segments, 1):
    segment_type = segment[:3]
    print(f"{i:2d}. [{segment_type}] {segment}")

print(f"\n{'-'*80}")
print(f"Total Segments: {len(segments)}")
print(f"\nSegment Types Present: {', '.join([seg[:3] for seg in segments])}")

# 🎯 HL7 ADT Synthetic Data Generator - Implementation Complete

## ✅ Phase 1 & 2 Enhancements Successfully Implemented

### Phase 1 (High Priority)
1. **DG1 Segment** - Diagnosis with ICD-10 codes ✅
   - 15 common ICD-10 codes (E11.9, I10, J44.9, I25.10, N18.3, etc.)
   - Diagnosis types: A (Admitting), W (Working), F (Final)

2. **Enhanced PID Segment** ✅
   - PID-10: Race (HL7 codes: 2106-3=White, 2054-5=Black, 2028-9=Asian, etc.)
   - PID-16: Marital Status (M/S/D/W/A)

3. **Enhanced PV1 Segment** ✅
   - PV1-10: Hospital Service (MED/SUR/OBS/CAR/NEU/ONC/ORT/PSY/PED)
   - PV1-44: Admit Date/Time (first A01/A04 event)
   - PV1-45: Discharge Date/Time (A03 events only)

### Phase 2 (Enhanced Realism)
4. **Extended PID Demographics** ✅
   - PID-17: Religion (CAT/BAP/PRO/JEW/MOS/OTH/NON)
   - PID-22: Ethnic Group (H=Hispanic, N=Non-Hispanic)
   - PID-23: Birth Place (City, State format)

5. **AL1 Segment(s) - Allergies** ✅
   - 30% of patients have allergies
   - 1-2 allergies per patient (50% have 2nd allergy)
   - Types: DA (Drug), FA (Food), MA (Misc), EA (Environmental)
   - Severity: MI/MO/SV (Mild/Moderate/Severe)
   - Common allergies: Aspirin, Penicillin, Peanuts, Latex, Shellfish, etc.

6. **PV2 Segment** ✅
   - Admit Reason (10 realistic reasons: "Chest pain", "Shortness of breath", etc.)
   - Expected Discharge Date (admit + 3-7 days)

---

## 🏗️ Architecture: Multi-UDF Approach (Option 3)

### Problem Solved
**Original Issue:** Monolithic UDF with 63 parameters caused Python worker crashes on serverless compute

**Solution:** Broke down into **11 smaller segment-specific UDFs**:
1. `build_msh_segment()` - Message Header (8 params)
2. `build_evn_segment()` - Event Type (2 params)
3. `build_pid_segment()` - Patient ID (17 params) ✨ Phase 1 & 2 fields
4. `build_nk1_segment()` - Next of Kin (4 params)
5. `build_pv1_segment()` - Patient Visit (9 params) ✨ Phase 1 fields
6. `build_pv2_segment()` - Visit Additional (2 params) ✨ Phase 2
7. `build_dg1_segment()` - Diagnosis (4 params) ✨ Phase 1
8. `build_al1_segments()` - Allergies (9 params) ✨ Phase 2
9. `build_gt1_segment()` - Guarantor (10 params)
10. `build_in1_segment()` - Insurance (20 params)
11. `build_in2_segment()` - Insurance Additional (2 params)

### Benefits
- ✅ **No Python worker crashes** - Each UDF is simple enough for serverless compute
- ✅ **Easier debugging** - Isolate issues to specific segments
- ✅ **Better maintainability** - Update individual segments independently
- ✅ **Modular design** - Add/remove segments without touching others

---

## 📊 Output Statistics
- **Total Messages:** 1,294 HL7 ADT messages
- **Event Types:** 10 types (A01-A13)
- **Segments per Message:** 10-12 segments (depending on allergies)
- **Message Structure:** MSH → EVN → PID → NK1 → PV1 → PV2 → DG1 → AL1(0-2) → GT1 → IN1 → IN2

---

## ⚠️ Known Limitations (Serverless Compute)
1. **No `.cache()/.persist()`** - Serverless doesn't support caching
   - Impact: Subsequent operations may recompute UDFs (slower performance)
   - Workaround: Write to Delta/Parquet once, read back for multiple uses

2. **UDF Performance** - Python UDFs are slower on serverless vs classic clusters
   - Current: ~1,294 messages generated successfully
   - For larger datasets (10K+ messages), consider classic cluster

---

## 🚀 Next Steps (Optional)
1. **Write to Files** - Export messages as individual `.hl7` files
2. **Validation** - Add HL7 spec compliance checking
3. **Volume Generation** - Scale up to 10K+ messages (consider classic cluster)
4. **Performance** - Switch to classic cluster for caching and faster UDF execution

In [0]:
# -------------------------------------------------------------------
# 5) Write as Parquet, then post-process to individual .hl7 files
# -------------------------------------------------------------------
# Two-stage approach for performance:
#   Stage 1: Write to Parquet in parallel (fast distributed write)
#   Stage 2: Read Parquet and split into individual .hl7 files (sequential)
#   Stage 3: Create 10 sample .txt files for testing
#
# Note: Threading on driver was tested but is slower due to dbutils.fs serialization.
#       Sequential writes are optimal for this use case.

import time

# Define subdirectories
parquet_stage_path = f"{output_path}/parquet_stage"
hl7_output_path = f"{output_path}/hl7_files"
txt_output_path = f"{output_path}/txt_files"

print(f"Stage 1: Writing to Parquet (parallel)...")
print(f"  Location: {parquet_stage_path}")

# Add filename column
messages_with_filename = messages_df.withColumn(
    "filename",
    F.concat(
        F.col("patient_id"),
        F.lit("_"),
        F.col("visit_id"),
        F.lit("_"),
        F.col("event_type"),
        F.lit("_"),
        F.col("event_datetime"),
        F.lit(".hl7")
    )
).select("filename", "hl7_message")

# Write as Parquet (distributed write - fast!)
messages_with_filename.write.mode("overwrite").parquet(parquet_stage_path)

print("✓ Stage 1 complete: Parquet written in parallel")

# Stage 2: Post-process Parquet to individual .hl7 files
print(f"\nStage 2: Post-processing to individual .hl7 files...")
print(f"  Output location: {hl7_output_path}")

# Read back from Parquet
messages_from_parquet = spark.read.parquet(parquet_stage_path)

# Collect and write individual files
messages_list = messages_from_parquet.collect()

print(f"  Writing {len(messages_list)} individual .hl7 files...")

# Ensure output directory exists
dbutils.fs.mkdirs(hl7_output_path)

# Write each message as individual file (sequential is fastest for UC volumes)
start_time = time.time()

for i, row in enumerate(messages_list, 1):
    filename = row["filename"]
    message = row["hl7_message"]
    file_path = f"{hl7_output_path}/{filename}"
    
    dbutils.fs.put(file_path, message, overwrite=True)
    
    # Show progress every 100 files
    if i % 100 == 0:
        print(f"    Progress: {i}/{len(messages_list)} files written...")

end_time = time.time()
write_time = end_time - start_time

print(f"\n✓ Stage 2 complete: {len(messages_list)} individual HL7 files created")
print(f"  Write time: {write_time:.1f}s ({len(messages_list)/write_time:.1f} files/sec)")

# Stage 3: Create 10 sample .txt files
print(f"\nStage 3: Creating 10 sample .txt files...")
print(f"  Output location: {txt_output_path}")

# Ensure txt output directory exists
dbutils.fs.mkdirs(txt_output_path)

# Take first 10 files and write them with .txt extension
for i, row in enumerate(messages_list[:10], 1):
    filename = row["filename"].replace(".hl7", ".txt")
    message = row["hl7_message"]
    file_path = f"{txt_output_path}/{filename}"
    
    dbutils.fs.put(file_path, message, overwrite=True)

print(f"✓ Stage 3 complete: 10 sample .txt files created")

# Clean up Parquet staging area
print(f"\nCleaning up staging area...")
dbutils.fs.rm(parquet_stage_path, recurse=True)
print(f"✓ Removed staging directory")

print(f"\n{'='*70}")
print(f"✓ SUCCESS: Wrote {len(messages_list)} HL7 files + 10 sample TXT files")
print(f"  Base path: {output_path}")
print(f"  HL7 files: {hl7_output_path} ({len(messages_list)} files)")
print(f"  TXT files: {txt_output_path} (10 files)")
print(f"  S3 Location: {storage_location_str}/{relative_path}/")
print(f"  Format: <patient_id>_<visit_id>_<event_type>_<timestamp>.[hl7|txt]")
print(f"  Performance: {write_time:.1f}s total, {len(messages_list)/write_time:.1f} files/sec")
print(f"{'='*70}")